In [34]:
event_hub_namespace='iesstsabbadbaa-grp-01-05'
#producer_eventhub_connection_str='Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=producer;SharedAccessKey=rNTPclgGrgNavcEOH5COxcPu07J+F1nFe+AEhMqyKOo=;EntityPath=group1_ride_status'
consumer_eventhub_connection_str='Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=consumer;SharedAccessKey=xDSdHYVG/gZqoh2YLZFCeNT3sjWaB9Bu5+AEhGxBbGg=;EntityPath=group1_ride_status'
eventhub_name='group1_ride_status'
consumer_eventhub_connection_str_requests='Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=s7PkoEvTFyJ5+RvaefGCQcluOBdpo7gZF+AEhCNenyU=;EntityPath=group1_passenger_requests'
eventhub_name_requests='group1_passenger_requests'

# Azure Blob
account_name='iesstsabbadbaa'
account_key='YiUournbsPdwTGVanY1bNSsUcy1Z7NUNoty1fo0SDxcHoe6gzTxeq7BlIG4c3owhT1IIhim2IsHF+AStMie8jw=='

container_name='streamed-data-group1'

In [35]:
import os
import subprocess

# Fetch the latest Spark 3.x.x version
# curl -s https://downloads.apache.org/spark/ → Fetches the Spark download page.
# grep -o 'spark-3\.[0-9]\+\.[0-9]\+' → Extracts only versions that start with spark-3. (ignoring Spark 4.x if it exists in the future).
# sort -V → Sorts the versions numerically.
# tail -1 → Selects the latest version.
spark_version = subprocess.run(
    "curl -s https://downloads.apache.org/spark/ | grep -o 'spark-3\\.[0-9]\\+\\.[0-9]\\+' | sort -V | tail -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

spark_version

'spark-3.5.5'

In [36]:
spark_release=spark_version
hadoop_version='hadoop3'

import os, time
start=time.time()
os.environ['SPARK_RELEASE']=spark_release
os.environ['HADOOP_VERSION']=hadoop_version
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/{spark_release}-bin-{hadoop_version}"

In [37]:
if True:
  # Run below commands in google colab
  !apt-get install openjdk-8-jdk-headless -qq > /dev/null # install Java8
  !wget -q http://apache.osuosl.org/spark/${SPARK_RELEASE}/${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz # download spark-3.3.X
  !tar xf ${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz # unzip it

  !pip install -q findspark # install findspark
  # findspark find your Spark Distribution and sets necessary environment variables

In [38]:
import findspark
findspark.init()

# Check the pyspark version
import pyspark
print(pyspark.__version__)

3.5.5


In [39]:
ride_status_schema = """
{
    "doc": "A ride status recording.",
    "name": "RideStatus",
    "namespace": "acme.status",
    "type": "record",
    "fields": [
        {"name": "ride_id", "type": "string"},
        {"name": "passenger_id", "type": "string"},
        {"name": "driver_id", "type": "string"},
        {"name": "ride_status", "type": {
            "type": "enum",
            "name": "RideStatusEnum",
            "symbols": ["completed", "cancelled"]
        }},
        {"name": "request_time", "type": "long"},
        {"name": "pickup_time", "type": "long"},
        {"name": "dropoff_time", "type": "long"},
        {"name": "ride_duration", "type": "float"},
        {"name": "pickup_location", "type": "string"},
        {"name": "dropoff_location", "type": "string"},
        {"name": "distance", "type": "float"},
        {"name": "price", "type": "float"},
        {"name": "tip", "type": "float"},
        {"name": "vehicle_type", "type": "string"},
        {"name": "cancellation_reason", "type": ["null", "string"], "default": null}
    ]
}
"""

passenger_request_schema = """
{
    "doc": "A passenger request recording.",
    "name": "PassengerRequest",
    "namespace": "acme.requests",
    "type": "record",
    "fields": [
        {"name": "request_id", "type": "string"},
        {"name": "timestamp", "type": "long"},
        {"name": "passenger_id", "type": "string"},
        {"name": "pickup_location", "type": "string"},
        {"name": "dropoff_location", "type": "string"},
        {"name": "distance", "type": "float"},
        {"name": "status", "type": {
            "type": "enum",
            "name": "RequestStatusEnum",
            "symbols": ["completed", "cancelled"]
        }},
        {"name": "payment_type", "type": ["null", "string"], "default": null}
    ]
}
"""

In [40]:
# JARs needed for Hadoop-compatible access in Azure Storage
jar_dependencies= ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
    "org.apache.spark:spark-avro_2.12:3.5.0",
    "org.apache.hadoop:hadoop-azure:3.3.1",          # Hadoop Azure connector
    "com.microsoft.azure:azure-storage:8.6.6"        # Azure Blob SDK dependency
])


In [41]:
from pyspark.sql import SparkSession
from pyspark.sql.avro.functions import from_avro

# Create a Spark session


spark = SparkSession \
    .builder \
    .appName("StreamingAVROFromKafka") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config("spark.jars.packages", jar_dependencies) \
    .config(f"fs.azure.account.key.{account_name}.blob.core.windows.net", account_key) \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()


In [42]:
# Kafka Configuration for reading from Kafka/Event Hub
# Kafka source will create a unique group id for each query automatically. The user can set the prefix of the automatically
# generated group.id’s via the optional source option groupIdPrefix, default value is “spark-kafka-source”.
# Consumer groups in Kafka/Event Hubs are typically defined explicitly by clients (Kafka consumers),
# but Spark Structured Streaming manages consumer offsets internally, without explicitly registering or creating a visible consumer group within Azure Portal.
# groupIdPrefix auto-generates consumer group IDs for Kafka internally, but these DO NOT appear explicitly as consumer groups within the Azure Portal under
# Event Hub Consumer Groups.

kafkaConfRides = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    # Below settins required if kafka is secured, for example when connecting to Azure Event Hubs:
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{consumer_eventhub_connection_str}";',

    "subscribe": eventhub_name, # subscribe to the entire topic
    "startingOffsets": "latest", # "latest", "earliest", '{"topic_name: {"0": 62821}}'
    # "startingOffsets": f'{{"{eventhub_name}": {{"0": 212350, "1": -1, "2": 212152, "3": -1}}}}', # "latest", "earliest", '{"topic_name: {"0": 212350, "1": -1, "2": 212152, "3": -1}}' -1: latest, -2: earliest

    # "assign": f'{{"{eventhub_name}": [0]}}', # to read from specific partitions use option: "assign": '{"topic_name": [0, 1]}'
    # "startingOffsets": f'{{"{eventhub_name}": {{"0": 212350}}}}', # "latest", "earliest", '{"topic_name: {"0": 62821}}' -1: latest, -2: earliest

    "enable.auto.commit": "true ",
    "groupIdPrefix": "Stream_Analytics_",
    "auto.commit.interval.ms": "5000"
}

# now passenger‐requests consumer config, pointing at the other hub
kafkaConfReqs = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism":     "PLAIN",
    "kafka.security.protocol":  "SASL_SSL",
    # this password must include EntityPath=group1_passenger_requests
    "kafka.sasl.jaas.config":   f'org.apache.kafka.common.security.plain.PlainLoginModule required \
username="$ConnectionString" \
password="{consumer_eventhub_connection_str_requests}";',
    "subscribe":                "group1_passenger_requests",
    "startingOffsets":          "latest",
    "enable.auto.commit":       "true",
    "groupIdPrefix":            "Stream_Analytics_",
    "auto.commit.interval.ms":  "5000"
}

print(kafkaConfRides)
print(kafkaConfReqs)

{'kafka.bootstrap.servers': 'iesstsabbadbaa-grp-01-05.servicebus.windows.net:9093', 'kafka.sasl.mechanism': 'PLAIN', 'kafka.security.protocol': 'SASL_SSL', 'kafka.sasl.jaas.config': 'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=consumer;SharedAccessKey=xDSdHYVG/gZqoh2YLZFCeNT3sjWaB9Bu5+AEhGxBbGg=;EntityPath=group1_ride_status";', 'subscribe': 'group1_ride_status', 'startingOffsets': 'latest', 'enable.auto.commit': 'true ', 'groupIdPrefix': 'Stream_Analytics_', 'auto.commit.interval.ms': '5000'}
{'kafka.bootstrap.servers': 'iesstsabbadbaa-grp-01-05.servicebus.windows.net:9093', 'kafka.sasl.mechanism': 'PLAIN', 'kafka.security.protocol': 'SASL_SSL', 'kafka.sasl.jaas.config': 'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="Endpoint=sb://iesstsabbadbaa-grp-01-05.servicebus.windows.net/;SharedAc

In [43]:
# Read from Event Hub using Kafka
df_rides = spark \
    .readStream \
    .format("kafka") \
    .options(**kafkaConfRides)

# Read from Event Hub using Kafka
df_reqs = spark \
    .readStream \
    .format("kafka") \
    .options(**kafkaConfReqs)

In [44]:
df_rides = df_rides.load()  # Start reading data from the specified Kafka topic
df_rides.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [45]:
df_reqs = df_reqs.load()  # Start reading data from the specified Kafka topic
df_reqs.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [46]:
# Deserialize the AVRO messages from the value column
df_rides = df_rides.select(from_avro(df_rides.value, ride_status_schema).alias("group1_ride_status"))

# Print the schema of the DataFrame
df_rides.printSchema()

root
 |-- group1_ride_status: struct (nullable = true)
 |    |-- ride_id: string (nullable = false)
 |    |-- passenger_id: string (nullable = false)
 |    |-- driver_id: string (nullable = false)
 |    |-- ride_status: string (nullable = false)
 |    |-- request_time: long (nullable = false)
 |    |-- pickup_time: long (nullable = false)
 |    |-- dropoff_time: long (nullable = false)
 |    |-- ride_duration: float (nullable = false)
 |    |-- pickup_location: string (nullable = false)
 |    |-- dropoff_location: string (nullable = false)
 |    |-- distance: float (nullable = false)
 |    |-- price: float (nullable = false)
 |    |-- tip: float (nullable = false)
 |    |-- vehicle_type: string (nullable = false)
 |    |-- cancellation_reason: string (nullable = true)



In [47]:
# Deserialize the AVRO messages from the value column
df_reqs = df_reqs.select(from_avro(df_reqs.value, passenger_request_schema).alias("group1_passenger_requests"))

# Print the schema of the DataFrame
df_reqs.printSchema()

root
 |-- group1_passenger_requests: struct (nullable = true)
 |    |-- request_id: string (nullable = false)
 |    |-- timestamp: long (nullable = false)
 |    |-- passenger_id: string (nullable = false)
 |    |-- pickup_location: string (nullable = false)
 |    |-- dropoff_location: string (nullable = false)
 |    |-- distance: float (nullable = false)
 |    |-- status: string (nullable = false)
 |    |-- payment_type: string (nullable = true)



In [48]:
 # Now we flatten the AVRO record to put each field in a separate column:
from pyspark.sql.functions import col

# flatten out every field in that struct
df_rides_flat = df_rides.select(
    col("group1_ride_status.ride_id"),
    col("group1_ride_status.passenger_id"),
    col("group1_ride_status.driver_id"),
    col("group1_ride_status.ride_status"),
    col("group1_ride_status.request_time"),
    col("group1_ride_status.pickup_time"),
    col("group1_ride_status.dropoff_time"),
    col("group1_ride_status.ride_duration"),
    col("group1_ride_status.pickup_location"),
    col("group1_ride_status.dropoff_location"),
    col("group1_ride_status.distance"),
    col("group1_ride_status.price"),
    col("group1_ride_status.tip"),
    col("group1_ride_status.vehicle_type"),
    col("group1_ride_status.cancellation_reason")
)

df_rides_flat.printSchema()


root
 |-- ride_id: string (nullable = true)
 |-- passenger_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- ride_status: string (nullable = true)
 |-- request_time: long (nullable = true)
 |-- pickup_time: long (nullable = true)
 |-- dropoff_time: long (nullable = true)
 |-- ride_duration: float (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- dropoff_location: string (nullable = true)
 |-- distance: float (nullable = true)
 |-- price: float (nullable = true)
 |-- tip: float (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- cancellation_reason: string (nullable = true)



In [49]:
# Now we flatten the AVRO record to put each field in a separate column:

# flatten out every field in the passenger‑requests struct
df_reqs_flat = df_reqs.select(
    col("group1_passenger_requests.request_id"),
    col("group1_passenger_requests.timestamp"),
    col("group1_passenger_requests.passenger_id"),
    col("group1_passenger_requests.pickup_location"),
    col("group1_passenger_requests.dropoff_location"),
    col("group1_passenger_requests.distance"),
    col("group1_passenger_requests.status"),
    col("group1_passenger_requests.payment_type")
)

df_reqs_flat.printSchema()

root
 |-- request_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- passenger_id: string (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- dropoff_location: string (nullable = true)
 |-- distance: float (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_type: string (nullable = true)



In [50]:
# Ride‑status sink paths
rides_output_path     = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/rides-output/"
rides_checkpoint_path = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/rides-checkpoint/"

# Passenger‑requests sink paths
reqs_output_path     = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/requests-output/"
reqs_checkpoint_path = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/requests-checkpoint/"

rides_output_path, rides_checkpoint_path, reqs_output_path, reqs_checkpoint_path


('wasbs://streamed-data-group1@iesstsabbadbaa.blob.core.windows.net/rides-output/',
 'wasbs://streamed-data-group1@iesstsabbadbaa.blob.core.windows.net/rides-checkpoint/',
 'wasbs://streamed-data-group1@iesstsabbadbaa.blob.core.windows.net/requests-output/',
 'wasbs://streamed-data-group1@iesstsabbadbaa.blob.core.windows.net/requests-checkpoint/')

In [51]:
# Ride‑status streaming write
rides_query = df_rides_flat.writeStream \
    .format("parquet") \
    .option("checkpointLocation", rides_checkpoint_path) \
    .option("path",               rides_output_path) \
    .queryName("rides_to_blob") \
    .trigger(processingTime="5 seconds") \
    .start()

# Passenger‑requests streaming write
reqs_query = df_reqs_flat.writeStream \
    .format("parquet") \
    .option("checkpointLocation", reqs_checkpoint_path) \
    .option("path",               reqs_output_path) \
    .queryName("requests_to_blob") \
    .trigger(processingTime="5 seconds") \
    .start()

In [52]:
# Get the list of active streaming queries
active_queries = spark.streams.active

# Print details about each active query
for query in active_queries:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Query Status: {query.status}")
    print(f"Is Query Active: {query.isActive}")
    print("-" * 50)


Query Name: requests_to_blob
Query ID: 4ca5069b-108b-4cee-ba18-8f96c56d3edf
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------
Query Name: rides_to_blob
Query ID: e8e939f2-e41a-40f8-871d-b9da417297d3
Query Status: {'message': 'Getting offsets from KafkaV2[Subscribe[group1_ride_status]]', 'isDataAvailable': False, 'isTriggerActive': True}
Is Query Active: True
--------------------------------------------------


In [53]:
#Only informative
"""
# Define a mapping of stream names → their checkpoint roots
checkpoint_paths = {
    "rides":    rides_checkpoint_path,
    "requests": reqs_checkpoint_path
}

for name, cp_path in checkpoint_paths.items():
    print(f"\n===== Offsets for stream: {name} =====")
    # Spark’s metadata lives under “sources/0/0” (we can read the entire folder
    # and Spark will concatenate any part files; usually there’s just one)
    df_off = spark.read.text(f"{cp_path}/sources/0")
    df_off.show(truncate=False)
"""

'\n# Define a mapping of stream names → their checkpoint roots\ncheckpoint_paths = {\n    "rides":    rides_checkpoint_path,\n    "requests": reqs_checkpoint_path\n}\n\nfor name, cp_path in checkpoint_paths.items():\n    print(f"\n===== Offsets for stream: {name} =====")\n    # Spark’s metadata lives under “sources/0/0” (we can read the entire folder\n    # and Spark will concatenate any part files; usually there’s just one)\n    df_off = spark.read.text(f"{cp_path}/sources/0")\n    df_off.show(truncate=False)\n'

In [54]:
#Only informative
"""
for progress in rides_query.recentProgress:
    print("==== Progress ====")
    print(f"Batch ID: {progress['batchId']}")
    print(f"Input rows: {progress['numInputRows']}")
    print(f"Processed time: {progress['timestamp']}")
    print(f"Sink: {progress['sink']}")
    print("===")
"""

'\nfor progress in rides_query.recentProgress:\n    print("==== Progress ====")\n    print(f"Batch ID: {progress[\'batchId\']}")\n    print(f"Input rows: {progress[\'numInputRows\']}")\n    print(f"Processed time: {progress[\'timestamp\']}")\n    print(f"Sink: {progress[\'sink\']}")\n    print("===")\n'

In [55]:
#Only informative
"""
for progress in reqs_query.recentProgress:
    print("==== Progress ====")
    print(f"Batch ID: {progress['batchId']}")
    print(f"Input rows: {progress['numInputRows']}")
    print(f"Processed time: {progress['timestamp']}")
    print(f"Sink: {progress['sink']}")
    print("===")
"""

'\nfor progress in reqs_query.recentProgress:\n    print("==== Progress ====")\n    print(f"Batch ID: {progress[\'batchId\']}")\n    print(f"Input rows: {progress[\'numInputRows\']}")\n    print(f"Processed time: {progress[\'timestamp\']}")\n    print(f"Sink: {progress[\'sink\']}")\n    print("===")\n'

In [56]:
# Spark's consumer offset is stored at checkpoint/offsets/
"""
checkpoint_files = spark.read.text(f"{rides_checkpoint_path}/offsets/").collect()
for row in checkpoint_files:
    print(row.value)
"""

'\ncheckpoint_files = spark.read.text(f"{rides_checkpoint_path}/offsets/").collect()\nfor row in checkpoint_files:\n    print(row.value)\n'

In [57]:
# Spark's consumer offset is stored at checkpoint/offsets/
"""
checkpoint_files = spark.read.text(f"{reqs_checkpoint_path}/offsets/").collect()
for row in checkpoint_files:
    print(row.value)
"""

'\ncheckpoint_files = spark.read.text(f"{reqs_checkpoint_path}/offsets/").collect()\nfor row in checkpoint_files:\n    print(row.value)\n'

In [58]:
# Spark's consumer offset is stored at checkpoint/offsets/
"""
from pyspark.sql import functions as F
from datetime import datetime
import json

# Number of recent offset files to show
N = 5

# Path to offsets
offset_path = f"{checkpoint_path}/offsets/*"

# Load all offset files and attach filename
df = spark.read.text(offset_path).withColumn("filename", F.input_file_name())

# Extract timestamp metadata from lines with "batchTimestampMs"
def extract_meta(row):
    try:
        data = json.loads(row.value)
        if "batchTimestampMs" in data:
            ts = data["batchTimestampMs"]
            readable_ts = datetime.utcfromtimestamp(ts / 1000).isoformat()
            return {
                "batchTimestampMs": ts,
                "timestamp": readable_ts,
                "filename": row.filename
            }
    except:
        return None

# Step 1: Collect and sort offset metadata
meta = df.rdd.map(extract_meta).filter(lambda x: x is not None).collect()
sorted_meta = sorted(meta, key=lambda x: x["batchTimestampMs"])
recent_meta = sorted_meta[-N:]  # take last N

# Step 2: For each of the recent N files, display its timestamp and full contents
for entry in recent_meta:
    file_path = entry["filename"]
    print(f"\n🗂 Offset file: {file_path}")
    print(f"🕒 Timestamp  : {entry['timestamp']}")
    print("📦 Content:")

    file_df = spark.read.text(file_path)
    file_df.show(truncate=False)
"""

'\nfrom pyspark.sql import functions as F\nfrom datetime import datetime\nimport json\n\n# Number of recent offset files to show\nN = 5\n\n# Path to offsets\noffset_path = f"{checkpoint_path}/offsets/*"\n\n# Load all offset files and attach filename\ndf = spark.read.text(offset_path).withColumn("filename", F.input_file_name())\n\n# Extract timestamp metadata from lines with "batchTimestampMs"\ndef extract_meta(row):\n    try:\n        data = json.loads(row.value)\n        if "batchTimestampMs" in data:\n            ts = data["batchTimestampMs"]\n            readable_ts = datetime.utcfromtimestamp(ts / 1000).isoformat()\n            return {\n                "batchTimestampMs": ts,\n                "timestamp": readable_ts,\n                "filename": row.filename\n            }\n    except:\n        return None\n\n# Step 1: Collect and sort offset metadata\nmeta = df.rdd.map(extract_meta).filter(lambda x: x is not None).collect()\nsorted_meta = sorted(meta, key=lambda x: x["batchTimes

In [59]:
"""progress = rides_query.lastProgress
import json
print(json.dumps(progress, indent=2))
"""

'progress = rides_query.lastProgress\nimport json\nprint(json.dumps(progress, indent=2))\n'

THIS STOPS THE PIPELINE

In [62]:
# Set to True and run cell when you want to stop your queries and Spark job.
if True:
  # Get the list of active streaming queries
  active_queries = spark.streams.active

# Print details about each active query
  for query in active_queries:
      query.stop()
      print(f"Query Name: {query.name}")
      print(f"Query ID: {query.id}")
      print(f"Query Status: {query.status}")
      print(f"Is Query Active: {query.isActive}")
      print("-" * 50)
  spark.stop()
  spark.sparkContext.stop()

Query Name: requests_to_blob
Query ID: 4ca5069b-108b-4cee-ba18-8f96c56d3edf
Query Status: {'message': 'Terminated with exception: Please see the cause for further information.', 'isDataAvailable': False, 'isTriggerActive': False}
Is Query Active: False
--------------------------------------------------
Query Name: rides_to_blob
Query ID: e8e939f2-e41a-40f8-871d-b9da417297d3
Query Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Is Query Active: False
--------------------------------------------------


extra

In [61]:
from pyspark.sql import SparkSession

# re‑use your existing SparkSession, or rebuild it with the same fs.azure.config
spark = SparkSession.builder.getOrCreate()

# adjust these if you used a different output folder
avro_path = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/rides-output/"

# load all Avro files (or point at a single file under that prefix)
df_avro = spark.read \
    .format("parquet") \
    .load(avro_path)

# inspect schema & a few rows
#df_avro.printSchema()
df_avro.show(10, truncate=False)


+-------+------------+---------+-----------+------------+-----------+------------+-------------+---------------+----------------+--------+-----+---+------------+-------------------+
|ride_id|passenger_id|driver_id|ride_status|request_time|pickup_time|dropoff_time|ride_duration|pickup_location|dropoff_location|distance|price|tip|vehicle_type|cancellation_reason|
+-------+------------+---------+-----------+------------+-----------+------------+-------------+---------------+----------------+--------+-----+---+------------+-------------------+
+-------+------------+---------+-----------+------------+-----------+------------+-------------+---------------+----------------+--------+-----+---+------------+-------------------+

